# M0: AV 3.0 Pipeline Overview

**Pipeline Position:** Overview & Reference Module  
**Input S3 Path:** N/A (reference only)  
**Output S3 Path:** N/A (reference only)  
**Source Repo:** [NVIDIA Cosmos Cookbook](https://github.com/nvidia-cosmos/cosmos-cookbook) — the post-training recipes this lab derives from. NVIDIA has since released [**Cosmos 3**](https://github.com/NVIDIA/Cosmos), and the Cookbook is now maintenance-only; this lab pins the Cosmos 2.5-era models and recipes it was built and verified against.  
**Instance:** ml.t3.medium (CPU only)

## Pipeline Architecture

This lab implements the **8-stage** end-to-end Physical AI data pipeline from the
[AWS + NVIDIA AV 3.0 blog](https://aws.amazon.com/blogs/industries/building-an-end-to-end-physical-ai-data-pipeline-for-autonomous-vehicle-3-0-on-aws-with-nvidia/),
plus a few supplementary modules (M6, M11, M12) that extend it. Everything derives
from two sources in the shared S3 bucket: the **nuScenes-mini dataset** and the
**pre-cached model weights** (`model-cache/`). Arrows show data flow.

```
 SHARED S3  (av30lab-shared-data-<acct>-<region>)
 ┌─────────────────────────────────────┐   ┌──────────────────────────────────────┐
 │ datasets/nuscenes-mini/             │   │ model-cache/  (pre-cached GPU models) │
 │   metadata tables + CAM_FRONT images│   │   cosmos-reason1/    → M2, M8          │
 └───────┬───────────────────┬─────────┘   │   cosmos-transfer2.5/→ M5              │
   M1     │                   │  M7        │   cosmos-predict2.5/ → M6              │
   reads  ▼                   ▼  reads CAM   │   alpamayo-1.5/      → M9              │
 ┌────────────┐         ┌──────────────────┐└──────────────────────────────────────┘
 │ M1         │Stage 1-2│ M7  Stage 6      │       (each GPU module s3-syncs its
 │ Data       │         │ Nerfstudio 3D    │        model folder at run time)
 │ Explore    │         │ Reconstruction   │
 │ CPU        │         │ g5.xlarge        │  ┌───────────────────────────────┐
 └─────┬──────┘         └──────────────────┘  │ M8   Stage 7                  │
       │ m1/  (+ nuScenes annotation tables)  │ Cosmos Reason LoRA SFT        │
       ├────────────────────────────────────▶ │ targets = nuScenes HUMAN      │
       │                                      │ labels, NOT m2/captions.json  │
       │                                      │ g6.24xlarge  → m8/            │
       │                                      └───────────────────────────────┘
       ▼
 ┌────────────┐  m2/    ┌────────────┐  m3/     ┌───────────────┐
 │ M2 Stage 3 │ caption │ M3 Stage 3 │ curated  │ (M3 fans out  │
 │ Cosmos     │────────▶│ Cosmos     │─────────▶│  to 4 modules │
 │ Reason     │         │ Curator    │          │  below)       │
 │ Captioning │         │ (filter)   │          └──────┬────────┘
 │ g5.12xlarge│         │ g5.12xlarge│                 │
 └─────┬──────┘         └────────────┘                 │
       │ m2/captions.json                              │
       ▼                                               │
 ┌────────────┐ Stage 4                                │
 │ M4         │ Search & Indexing                      │
 │ OpenSearch │ blog Path A, on                        │
 │ Serverless │ Serverless; vector-                    │
 │ k-NN   CPU │ only (no hybrid)                       │
 └────────────┘                                        │
        ┌──────────────┬──────────────┬────────────────┘
        ▼              ▼              ▼              ▼
  ┌──────────┐  ┌──────────┐  ┌────────────┐  ┌──────────────┐
  │ M5       │  │ M6       │  │ M9         │  │ M12           │
  │ Stage 5  │  │ Stage 5  │  │ Stage 7    │  │ Stages 3/5/7 │
  │ Cosmos   │  │ (ext)    │  │ Alpamayo   │  │ (ext)        │
  │ Transfer │  │ Cosmos   │  │ VLA        │  │ HyperPod     │
  │ (Weather │  │ Predict  │  │ inference  │  │ Distributed  │
  │  Aug)    │  │ (Scenario│  │            │  │ Training     │
  │ p4d.24xl │  │  Gen)    │  │ p4d.24xl   │  │ p4d (concept)│
  └──────────┘  │ p4d.24xl │  └─────┬──────┘  └──────────────┘
                └──────────┘        │ m9/ policy
                                    ▼
                              ┌────────────┐ Stage 8
                              │ M10         │ Software-in-the-Loop
                              │ AlpaSim    │ Testing (closed-loop)
                              │ Eval       │
                              │ g5.12xlarge│
                              └────────────┘

 Orchestration:  ┌──────────────────┐
                 │ M11 (ext)        │ reads m1/ , runs M1→M2→M3→M5
                 │ Pipeline Autom.  │ as one SageMaker Pipeline
                 │ CPU              │
                 └──────────────────┘
```

**Blog stage → module mapping** (blog calls these 8 items *stages*, grouped into 4
*phases*: Ingest, Data Processing, Train, Validate):

| Blog Stage | Module(s) |
|-----------|-----------|
| 1. Ingest to cloud | M1 |
| 2. Data quality & sensor extraction | M1 |
| 3. Data curation (blog: **Cosmos Curator** on HyperPod+SLURM — split, transcode, caption, embed) | M2 (caption), M3 (split+transcode+motion filter, **NeMo** Curator), M4 (embed) |
| 4. Search and indexing (blog: OpenSearch **Service** Path A, or Cosmos Dataset Search Path B) | M4 — Path A on OpenSearch **Serverless** |
| 5. Data augmentation (blog: Cosmos Transfer on **EC2 + DCV**, ~65 GB GPU → G7e) | M5 — Studio notebook, 24 GB/card default at 480p |
| 6. Neural reconstruction (blog: **NuRec**) | M7 — Nerfstudio / gsplat |
| 7. Model training (blog: Alpamayo) | **M8** (LoRA SFT of Cosmos Reason), M9 (Alpamayo *inference*) |
| 8. Software-in-the-loop testing (AlpaSim) | M10 |
| *extension* — synthetic scenario gen (Cosmos Predict) | M6 |
| *extension* — distributed training scale-up (HyperPod) | M12 |
| *extension* — pipeline orchestration | M11 |

**Reading the diagram**
- **Two sources** in the shared bucket: `datasets/nuscenes-mini/` and `model-cache/`.
- **Core path:** M1 → M2 → M3 (ingest → caption → curate).
- **M3 fans out** to M5, M6, M9, M12 — each consumes `m3/`.
- **Model cache:** M2/M5/M6/M8/M9 each `aws s3 sync` their model folder from
  `model-cache/` at run time (Cosmos Reason 1 / Transfer 2.5 / Predict 2.5 / Alpamayo 1.5).
- **M4** branches off **M2** (`m2/captions.json`); **M10** consumes **M9** (`m9/`).
- **The search-driven loop is not wired here.** In the blog, Stage 5 *queries the
  Stage-4 index* to pick which clips to augment. In this lab M5/M6 read `m3/`
  directly and M4 is a leaf, so you can run either without the other.
- **M7** reads the **nuScenes CAM_FRONT images directly** (independent branch).
- **M8** reads `m1/manifest.json` plus the **nuScenes annotation tables** — its
  training targets are human labels, deliberately not M2's captions (training
  Cosmos Reason on its own output would make the loss self-referential).
- **M11** orchestrates the M1→M2→M3→M5 sub-pipeline as one SageMaker Pipeline.


In [ ]:
"""NVIDIA AV Tool Availability for AV 3.0 Blueprint Lab

Source URLs and HuggingFace IDs below were verified to resolve. Two notes on why
some links may look unexpected:

  * The **Cosmos Cookbook** is its own repo (nvidia-cosmos/cosmos-cookbook), NOT a
    subdirectory of NVIDIA/Cosmos, and its recipes live under docs/recipes/.
  * NVIDIA/Cosmos is now the **Cosmos 3** home. The Cosmos 2.5-era code this lab
    uses lives in the separate nvidia-cosmos/cosmos-{transfer,predict}2.5 repos,
    which are maintenance-only but are the only place that code exists.

Model IDs match what scripts/cache_models.sh actually pre-caches — keep them in
sync, since a wrong-but-plausible ID fails at load time rather than here.
"""
import pandas as pd

tools_data = [
    {
        "Tool": "Cosmos Predict 2.5",
        "Version": "2.5",
        "Status": "Available",
        "License": "NVIDIA Open Model License",
        "Source URL": "https://github.com/nvidia-cosmos/cosmos-predict2.5",
        "How to Access": "HuggingFace: nvidia/Cosmos-Predict2.5-2B (gated; used in M6)"
    },
    {
        "Tool": "Cosmos Transfer 2.5",
        "Version": "2.5",
        "Status": "Available",
        "License": "NVIDIA Open Model License",
        "Source URL": "https://github.com/nvidia-cosmos/cosmos-transfer2.5",
        "How to Access": "HuggingFace: nvidia/Cosmos-Transfer2.5-2B (gated; used in M5)"
    },
    {
        "Tool": "Cosmos Reason 1",
        "Version": "1.0",
        "Status": "Available",
        "License": "NVIDIA Open Model License",
        "Source URL": "https://github.com/nvidia-cosmos/cosmos-cookbook/tree/main/docs/recipes/post_training/reason1",
        "How to Access": "HuggingFace: nvidia/Cosmos-Reason1-7B (ungated; used in M2 and M8)"
    },
    {
        "Tool": "Cosmos Reason 2",
        "Version": "2.0",
        "Status": "Available",
        "License": "NVIDIA Open Model License",
        "Source URL": "https://github.com/nvidia-cosmos/cosmos-cookbook/tree/main/docs/recipes/post_training/reason2",
        "How to Access": "HuggingFace: nvidia/Cosmos-Reason2-8B (gated; M9/M10 Alpamayo VLM backbone). Also 2B / 32B"
    },
    {
        # The blog's Stage 3 tool. NOT what this lab runs — listed so the
        # substitution is visible rather than hidden behind a merged row.
        "Tool": "Cosmos Curator (blog's Stage 3 tool)",
        "Version": "n/a",
        "Status": "Not used by this lab",
        "License": "Apache 2.0",
        "Source URL": "https://github.com/nvidia-cosmos/cosmos-curate",
        "How to Access": "Docker + SLURM on SageMaker HyperPod. Out of scope here; M3 substitutes NeMo Curator"
    },
    {
        # A SEPARATE NVIDIA project from cosmos-curate, despite the similar name.
        "Tool": "NeMo Curator (M3's actual tool)",
        "Version": "0.8+",
        "Status": "Available",
        "License": "Apache 2.0",
        "Source URL": "https://github.com/NVIDIA-NeMo/Curator",
        "How to Access": "pip install nemo-curator (used in M3: split + transcode + motion filter only)"
    },
    {
        "Tool": "Alpamayo 1.5 (VLA)",
        "Version": "1.5",
        "Status": "Available",
        "License": "NVIDIA Open Model License (non-commercial)",
        "Source URL": "https://github.com/NVlabs/alpamayo1.5",
        "How to Access": "HuggingFace: nvidia/Alpamayo-1.5-10B (gated; used in M9)"
    },
    {
        "Tool": "AlpaSim (closed-loop sim)",
        "Version": "0.96.0",
        "Status": "Preview",
        "License": "Apache-2.0 (code) / NVIDIA AV NuRec Dataset License (scenes)",
        "Source URL": "https://github.com/NVlabs/alpasim",
        "How to Access": "Public repo; NuRec scenes gated on HuggingFace (nvidia/PhysicalAI-Autonomous-Vehicles-NuRec). Used in M10"
    },
    {
        "Tool": "Physical AI Datasets (nuScenes)",
        "Version": "1.0",
        "Status": "Available",
        "License": "CC BY-NC-SA 4.0",
        "Source URL": "https://www.nuscenes.org/",
        "How to Access": "Pre-loaded in shared S3 bucket (mini split)"
    },
    {
        "Tool": "Cosmos Tokenizer",
        "Version": "1.0",
        "Status": "Available (repo archived Feb 2025)",
        "License": "NVIDIA Open Model License",
        "Source URL": "https://github.com/NVIDIA/Cosmos-Tokenizer",
        "How to Access": "HuggingFace: nvidia/Cosmos-Tokenizer-* (not used directly by this lab)"
    },
    {
        "Tool": "DRIVE Sim (Omniverse)",
        "Version": "2024.2",
        "Status": "Enterprise Only",
        "License": "NVIDIA Enterprise",
        "Source URL": "https://developer.nvidia.com/drive/simulation",
        "How to Access": "NVIDIA Enterprise subscription required"
    },
    {
        "Tool": "Cosmos World Foundation Model (WFM)",
        "Version": "3 (current) / 1.0 (legacy)",
        "Status": "Available",
        "License": "NVIDIA Open Model License",
        "Source URL": "https://github.com/NVIDIA/Cosmos",
        "How to Access": "Now Cosmos 3 (Super 64B / Nano 16B / Edge 4B). This lab pins the 2.5-era models above"
    }
]

df_tools = pd.DataFrame(tools_data)
print("=" * 80)
print("NVIDIA AV 3.0 Tool Availability Matrix")
print("=" * 80)
df_tools.style.set_properties(**{'text-align': 'left'})
df_tools

## AWS Service Mapping per Module (blog 8-stage)

| Module | Pipeline Stage | AWS Service | Instance Type | Purpose |
|--------|---------------|-------------|---------------|---------|
| M1 | Data Collection (Stage 1-2) | S3 + SageMaker Studio | ml.t3.medium | Browse & explore nuScenes-mini |
| M2 | AV Captioning (Stage 3) | SageMaker JupyterLab | ml.g5.12xlarge | Cosmos Reason 1 inference |
| M3 | Data Curation (Stage 3) | SageMaker JupyterLab | ml.g5.12xlarge | Clip split + transcode + motion filter; carries M2's captions through (no dedup) |
| M4 | Semantic Search (Stage 4) | SageMaker + OpenSearch Serverless | ml.t3.medium | Blog Stage 4 **Path A** (OpenSearch), on Serverless + vector-only k-NN; Path B (Cosmos Dataset Search on EKS) not covered |
| M5 | Weather Augmentation (Stage 5) | SageMaker JupyterLab | ml.g5.12xlarge | Cosmos Transfer 2.5 |
| M6 | Scenario Generation (Stage 5, ext) | SageMaker JupyterLab | ml.g5.12xlarge | Cosmos Predict 2.5 |
| M7 | 3D Reconstruction (Stage 6) | SageMaker JupyterLab | ml.g5.xlarge | Nerfstudio neural radiance fields *(training cell known-limited)* |
| M8 | Model Training (Stage 7) | SageMaker JupyterLab | ml.g5.12xlarge | Cosmos Reason LoRA SFT on nuScenes **human** labels |
| M9 | VLA Inference (Stage 7) | SageMaker JupyterLab | ml.g5.12xlarge | Alpamayo 1.5 inference |
| M10 | Closed-Loop Eval (Stage 8) | SageMaker JupyterLab | **ml.t3.medium (CPU)** | Visualizes an AlpaSim eval; real sim runs on a GPU EC2 host |
| M11 | Orchestration | SageMaker Pipelines | **ml.t3.medium notebook → ml.m5.xlarge steps** | Automate M1→M2→M3→M5 as one SageMaker Pipeline |
| M12 | Distributed Training (Stages 3/5/7, ext) | SageMaker Training | **ml.t3.medium notebook → ml.m5.xlarge×2 job** | Submits a real 2-node DDP job; HyperPod is conceptual |

\* **M10 and M12 notebooks are CPU** even though they're about GPU-scale ideas: M10
visualizes a closed-loop simulation the admin ran on a GPU EC2 host; M12 *submits* a
real 2-node `torch.distributed` job that runs on separate managed `ml.m5.xlarge`
instances. M12's HyperPod *cluster* is separate infrastructure — the notebook
demonstrates the distributed-training pattern, not a live HyperPod cluster.
See docs/en/HYPERPOD_M12.md and docs/en/ALPASIM_M10.md.

**Shared Infrastructure:**
- Shared data bucket: `s3://av30lab-shared-data-{account_id}-{region}/` (datasets + model cache)
- User workspace bucket: `s3://av30lab-user-workspace-{account_id}-{region}/users/{profile}/`
- IAM: SageMaker execution role with scoped S3, ECR, and CloudWatch access
- Network: SageMaker Studio domain in PublicInternetOnly mode
- GPU modules require the **SageMaker Distribution GPU** image (selected automatically
  when a GPU instance is chosen)


## Module Overview

| Module | Name | Instance | Blog Stage | Reads | Description |
|--------|------|----------|-------|-------|-------------|
| M0 | Pipeline Overview | ml.t3.medium | Overview | — | Architecture & tool reference |
| M1 | Data Exploration | ml.t3.medium | 1-2 | nuScenes-mini | Explore the nuScenes-mini dataset → `m1/` |
| M2 | Cosmos Reason Captioning | ml.g5.12xlarge | 3 | `m1/` | AV video captioning (Cosmos Reason 1) → `m2/` |
| M3 | Cosmos Curator | ml.g5.12xlarge | 3 | `m2/` | Data quality filtering & curation → `m3/` |
| M4 | OpenSearch Semantic Search | ml.t3.medium | 4 | `m2/` | Video clip retrieval (embeddings + k-NN) → `m4/` |
| M5 | Cosmos Transfer — Weather Aug | ml.g5.12xlarge | 5 | `m3/` | Weather-augmented clips (Cosmos Transfer 2.5) → `m5/` |
| M6 | Cosmos Predict — Scenario Gen | ml.g5.12xlarge | 5 (ext) | `m3/` | Synthetic traffic scenarios (Cosmos Predict 2.5) → `m6/` |
| M7 | Nerfstudio 3D Reconstruction | ml.g5.xlarge | 6 | nuScenes CAM_FRONT | Neural radiance field reconstruction → `m7/` *(training cell known-limited)* |
| M8 | Cosmos Reason LoRA SFT | ml.g5.12xlarge | 7 | `m1/` + nuScenes annotation | LoRA fine-tune on **human** labels (not M2's captions) → `m8/` |
| M9 | Alpamayo VLA Inference | ml.g5.12xlarge | 7 | `m3/` | Vision-Language-Action policy inference → `m9/` |
| M10 | AlpaSim Closed-Loop Eval | ml.t3.medium (CPU) | 8 | `m9/` | Visualizes a real AlpaSim eval (sim runs on a GPU EC2 host) → `m10/` |
| M11 | Pipeline Automation | ml.t3.medium | — (ext) | `m1/` | Orchestrate M1→M2→M3→M5 as one SageMaker Pipeline → `m11/` |
| M12 | HyperPod Distributed Training | ml.t3.medium (CPU) | 3/5/7 (ext) | `m3/` | Submits a real 2-node DDP job on `ml.m5.xlarge`×2 → `m12/` |

\* M10/M12 notebooks are **CPU**: M10 visualizes an admin-run AlpaSim eval; M12 submits a
real 2-node `torch.distributed` job to separate `ml.m5.xlarge` instances. M12's
HyperPod cluster is separate infrastructure (demonstrated conceptually).

**GPU notebook modules (M2, M3, M5–M9)** launch on GPU instances; the workshop selects
the matching SageMaker Distribution **GPU image** automatically. CPU modules (M0, M1, **M4**,
**M10**, M11, **M12**) run on `ml.t3.medium`. (M10's real simulation and M12's real
training job run on GPU/managed compute *outside* the notebook.)

**Cost note:** the biggest drivers are the GPU modules M5/M6/M8/M9. Their verified
default is `ml.g5.12xlarge` (~$7.09/hr); stepping up to `ml.p4d.24xlarge` for
full-resolution output costs ~$25.25/hr, and `ml.g7e.2xlarge` reaches the same
quality tier on one 96 GB card for ~$4.20/hr (needs its own quota). M12's managed training job (`ml.m5.xlarge`×2) and M11's pipeline steps are
CPU and cost cents per run. Idle apps auto-shut down after 3 hours; stop your space
when done to avoid charges.

---

*The module numbers follow the blog's 8-stage order, so running them in sequence
walks the pipeline as the blog describes it. Proceed to
**M1_Data_Exploration.ipynb** to begin.*
